# B. 독립변수 16개 추출 버전(t_test + logistic regression)


## 1. 라이브러리 설치

In [ ]:
# Colab에서 실행할 때만 아래 주석을 풀고, 프로젝트 폴더 경로를 맞춰 주세요.
# (로컬 Jupyter / VS Code에서는 이 노트북이 있는 폴더에서 그대로 실행)
# from google.colab import drive
# drive.mount('/content/drive')
# %cd /content/drive/MyDrive/<프로젝트 폴더>

## 2. 데이터 불러오기 및 탐색 함수


In [ ]:
import numpy as np
import pandas as pd

from config import (
    SEEDS, QUBIT_RANGE, SELECTION_METHODS, ENCODINGS, REDUCTIONS,
)
from src.common import (
    load_raw, run_eda, preprocess, build_df16, split16, iterate_splits,
)
from config import VQC_QUBIT_RANGE
from src.common import selection_stability
from src.classical import run_classical_models
from src.metrics import save_result, save_sweep, summarize_results
from src.vqc import vqc_sweep, vqc_reupload_sweep

NOTEBOOK = "ind16"



# 데이터: data/FILE2_중공업.xls (config.DATA_PATH)
df = load_raw()
df.tail(5)

## 3. 전체 탐색 실행


In [ ]:
# 탐색 함수와 출력 내용은 src/common.py 의 run_eda
x, dates, tables, summary = run_eda(df)

## 4. 전처리

### 기존 전처리

In [ ]:
# 전처리 클래스와 파이프라인은 src/common.py 의 preprocess
prep = preprocess(df)

### 독립변수 16개 추출

In [ ]:
# 148개 전처리 결과에서 16개 변수만 뽑아 df16 만들기 (src/common.py 의 build_df16)
df16 = build_df16(prep).df16
df16.head()

## 5. 모델링

### 실험 설계: 변수 선택 × 인코딩 × 큐비트 수 × seed

`config.py`의 목록을 모두 조합해 반복한다.

| 설정 | 내용 | 기본값 |
|---|---|---|
| `SELECTION_METHODS` | 변수 선택 방법 (seed마다 **훈련셋에서만** 선택) | `["fixed16"]` |
| `ENCODINGS` | 양자 인코딩 (basic / angle / iqp / dense_angle / amplitude) | `["basic"]` |
| `REDUCTIONS` | 큐비트보다 입력이 많을 때 줄이는 방법 (pca / select) | `["pca"]` |
| `QUBIT_RANGE` | 큐비트 수 | 1 ~ 16 |
| `SEEDS` | 반복 seed (분할 · 전처리 · 선택 · 모델 난수가 모두 바뀜) | 5개 |

seed마다 다시 하는 것: train/valid/test 분할(60/20/20, 층화) → 전처리 fit → 변수 후보(148개 + 업력) → 변수 선택 → 인코딩 fit → 모델 학습.

- 결과는 조합 하나가 끝날 때마다 `results/results.csv`에 저장되므로, 중간에 멈춰도 그때까지의 결과는 남는다.
- 마지막 셀에서 **평균 ± 표준오차**(95% 신뢰구간 포함)로 요약한다.
- 실행 시간은 조합 수에 비례한다. 먼저 목록을 줄여서 돌려 본다 (예: `SEEDS = [42]`, `QUBIT_RANGE = [4, 8, 16]`).
- `fixed16`은 전체 데이터로 고른 기존 목록이라 테스트 정보가 섞여 있다. 다른 방법(훈련셋에서만 선택)과의 차이가 선택 편향의 크기다.


### 고전 모델 (XGBoost · Random Forest · SVM · 로지스틱 회귀)

| 모델 | 하이퍼파라미터 선택 (검증셋 ROC-AUC) |
|---|---|
| XGBoost | `config.XGB_GRID` (max_depth, learning_rate, min_child_weight, reg_lambda), 트리 개수는 검증셋 조기 종료 |
| Random Forest | `config.RF_GRID` (트리 500개, max_depth, min_samples_leaf, max_features) |
| SVM (RBF) | 표준화 후 `config.SVM_GRID` (C, gamma) |
| 로지스틱 회귀 | 표준화 후 `config.LOGISTIC_GRID` (C, L1/L2 규제) |

- 입력은 선택된 변수 16개 그대로다 (PCA 없음). 변수 선택은 `config.SELECTION_METHODS`를 따른다 (지금은 기존 방식 `fixed16`).
- 테스트셋은 최종 평가에만 쓴다. 고른 값이 탐색 범위의 끝이면 경고가 출력된다.


#### 데이터 분할

In [ ]:
# 기준 seed(config.SEED) 분할 확인: 148개 버전과 같은 분할인지 내부에서 검증
# (실험 셀에서는 seed마다 prepare_split 으로 다시 나누고 변수도 다시 고른다)
sp = split16(df16, prep)

#### 학습 · 평가 (변수 선택 × seed)


In [ ]:
# 모델 정의와 하이퍼파라미터 탐색은 src/classical.py
selected_by = {}   # (변수 선택 방법, seed) → 선택된 변수

for data in iterate_splits(df, SELECTION_METHODS, SEEDS):
    selected_by[(data.selection, data.seed)] = data.selected

    for name, (results, params) in run_classical_models(data.X, data.Y, seed=data.seed).items():
        save_result(
            NOTEBOOK, name, results, seed=data.seed, selection=data.selection,
            params={**params, "features": data.selected},
        )

In [ ]:
# ============================================================
# 고전 모델 테스트 성능 (평균 ± 표준오차)
# ============================================================

classical_table, _ = summarize_results(notebook=NOTEBOOK)
classical_table = classical_table.xs("-", level="encoding")

print("[고전 모델 테스트 성능]")
print(classical_table.to_string())

# 변수 선택 방법이 2개 이상이면 선택 안정성(seed끼리 평균 Jaccard, 1 = 매번 같음)
if len(SELECTION_METHODS) > 1 and len(SEEDS) > 1:
    stability = pd.Series({
        method: selection_stability({seed: selected_by[(method, seed)] for seed in SEEDS})
        for method in SELECTION_METHODS
    }, name="선택 안정성 (Jaccard)")

    print("\n[선택 안정성]")
    print(stability.round(3).to_string())

### VQC (변수 선택 × 인코딩 × 1~8큐비트 × seed)

**양자 인코딩** (`config.ENCODINGS`, 양자 모델에만 적용)

| 인코딩 | 양자 상태 | 입력 수 d (n큐비트) |
|---|---|---|
| basic | 기존 방식: QSVM은 IQP 특징 맵, VQC는 RY 각도 | n |
| angle | 큐비트마다 RY(θ), 얽힘 없음 | n |
| iqp | H → Z·ZZ 위상 반복 (인접 큐비트 얽힘), QSVM·VQC 공통 | n |
| dense_angle | 큐비트 하나에 변수 2개: RY(θ₁) 후 RZ(θ₂) | 2n (n ≤ 8) |
| amplitude | 입력 벡터를 정규화해 상태 진폭으로 사용 | 2ⁿ (n ≤ 4) |

**입력 차원 맞추기** (`config.REDUCTIONS`): 선택된 변수 16개를 d개로 줄인다.

| 방법 | 처리 |
|---|---|
| pca | 표준화 → PCA d개 성분 → 성분 표준화 (기존 방식) |
| select | 변수 선택 순위 상위 d개 → 표준화 |

- d = 16이면 줄이지 않고 표준화만 한다. 각도형 인코딩은 θ = π·tanh(z/2)로 바꾼다.
- 쓸 수 없는 큐비트 수(예: amplitude 5큐비트 이상)는 자동으로 건너뛴다.
- basic + pca에서 n = 8은 기존 8큐비트(PCA 8개), n = 16은 기존 16큐비트 입력과 같다.
- 고전 모델은 PCA 없이 선택된 변수 16개를 그대로 쓴다.

| 회로 | 구성 |
|---|---|
| 임베딩 | 인코딩에 따라 RY 각도 / IQPEmbedding / RY + RZ / AmplitudeEmbedding |
| 학습 회로 | StronglyEntanglingLayers 2층 |
| 출력 | 각 큐비트 Pauli-Z 기대값 → Linear → logit |
| 학습 | Adam(lr 0.01), 배치 64, 최대 100 epoch, 검증 loss 기준 조기 종료(patience 15) |

- 하이퍼파라미터는 `config.py`의 VQC 항목에 있다.
- VQC는 큐비트 수를 `config.VQC_QUBIT_RANGE`(1~8)로 따로 둔다. 시뮬레이션 학습은 큐비트마다 시간이 약 2배씩 늘어서, 기존 8큐비트(PCA 8개)도 Colab에서 약 4.5분이 걸렸다.


In [ ]:
# 회로, 학습 루프, 평가는 src/vqc.py (verbose=True 이면 epoch별 로그 출력)
for data in iterate_splits(df, SELECTION_METHODS, SEEDS):
    for encoding in ENCODINGS:
        for reduction in REDUCTIONS:
            summary, runs = vqc_sweep(
                VQC_QUBIT_RANGE, data.X, data.Y,
                encoding=encoding, reduction=reduction, seed=data.seed,
            )

            save_sweep(
                NOTEBOOK, "VQC", runs, seed=data.seed,
                selection=data.selection, encoding=f"{encoding}/{reduction}",
                params_of=lambda run: {"best_epoch": run["info"]["best_epoch"]},
            )

### VQC + data re-uploading (변수 선택 × 인코딩 × 1~8큐비트 × seed)

같은 입력을 2번 인코딩하고(`config.N_REUPLOADS`), 인코딩마다 학습 회로를 붙인다. 나머지 설정은 VQC와 같다. amplitude 인코딩은 회로 맨 앞에서만 쓸 수 있어서 건너뛴다.


In [ ]:
for data in iterate_splits(df, SELECTION_METHODS, SEEDS):
    for encoding in ENCODINGS:
        for reduction in REDUCTIONS:
            summary, runs = vqc_reupload_sweep(
                VQC_QUBIT_RANGE, data.X, data.Y,
                encoding=encoding, reduction=reduction, seed=data.seed,
            )

            save_sweep(
                NOTEBOOK, "VQC+drc", runs, seed=data.seed,
                selection=data.selection, encoding=f"{encoding}/{reduction}",
            )

## 모델 성능 비교

In [ ]:
# ============================================================
# 요약: 평균 ± 표준오차 (테스트셋)
# ============================================================

summary_table, summary_full = summarize_results(notebook=NOTEBOOK)

print(f"[{NOTEBOOK}: 테스트 성능, 평균 ± 표준오차]")
print(summary_table.to_string())

# 지표별 n, mean, std, se, 95% 신뢰구간 전체 표
summary_full.round(4)

In [ ]:
# 참고: 다른 노트북 결과 (results/results.csv, 테스트셋, 평균 ± 표준오차)
reference_table, _ = summarize_results(exclude_notebook=NOTEBOOK)

print("[참고: 다른 노트북 결과]")
print(reference_table.to_string())

## 연구 보고서

> **참고:** 아래 보고서는 8큐비트와 16큐비트만, seed 하나로 실험하던 **이전 실험(리팩터링 전)** 기록이다.
> 현재 코드는 변수 선택 방법 × 인코딩 × 1~16큐비트 × 여러 seed로 실험한다.


2026년 10월 31일

#### 1. 실험 개요

| 항목 | 설정 |
|---|---|
| 데이터 수 | 3,140개 |
| Train / Validation / Test | 1,884 / 628 / 628 |
| 독립변수 | 16개 |
| 타깃 변수 | 정상/폐업 (이진분류) |
| 타깃 매핑 | 원본 코드 1 → 0, 원본 코드 2 → 1 |
| 데이터 분할 | 60% / 20% / 20% |
| 클래스 비율 | 50:50 |
| PCA 차원 | 8개 |
| PCA 누적 설명분산 비율 | 74.62% |
| 양자회로 | 8큐비트 |

**독립변수 16개 구성**

1. 영업이익률
2. 재료비/매출액
3. 감가상각률
4. 금융비용대총부채비율
5. 금융비용대총비용비율
6. EBITDA/총차입금
7. 차입금의존도
8. 장단기대여금/총자산
9. 법인세부담률
10. 순운전자본대총자본비율
11. 유동자산구성비율
12. 유보액대총자산비율
13. 재고자산대유동자산비율
14. 매입채무회전기간
15. 자본금회전율(회)
16. 업력

#### 2. 모델 구성

**공통 전처리**

- 극단값 제한: 훈련셋 기준 1% / 99% 분위수
- 결측값 처리: 중앙값 대치
- 스케일링: RobustScaler
- 검증 및 테스트 데이터는 훈련 데이터에서 학습한 전처리 객체로 transform만 적용
- 최종 입력 데이터: 16개 변수

**A. XGBoost (독립변수 16개 입력)**

| 항목 | 설정 |
|---|---|
| 목적함수 | binary:logistic |
| 트리 수 | 최대 2,000개 |
| 최적 트리 수 | 272개 |
| 조기 종료 | 검증 Log Loss 기준 100회 |
| learning_rate | 0.03 |
| max_depth | 3 |
| min_child_weight | 5 |
| subsample / colsample_bytree | 0.8 / 0.8 |
| reg_alpha / reg_lambda | 0.1 / 5.0 |
| scale_pos_weight | 1.0 |
| tree_method | hist |
| random_state | 42 |

**B. PCA-XGBoost (PCA 8개 입력)**

- 입력: 16개 독립변수
- StandardScaler → PCA(8개 성분)
- PCA는 훈련셋에서만 fit
- 누적 설명분산 비율: 74.62%

| 항목 | 설정 |
|---|---|
| n_estimators | 300 |
| max_depth | 4 |
| learning_rate | 0.05 |
| subsample | 0.8 |
| colsample_bytree | 0.8 |
| objective | binary:logistic |
| eval_metric | logloss |
| random_state | 42 |
| 조기 종료 | 미적용 |

주의: A와 B의 하이퍼파라미터가 동일하지 않음.

**C. VQC (8큐비트)**

| 단계 | 구성 |
|---|---|
| 입력 | 16개 변수 |
| 차원 축소 | VarianceThreshold → StandardScaler → PCA 8개 성분 |
| 성분 표준화 | StandardScaler |
| 각도 변환 | θ = π·tanh(z/2) |
| 인코딩 | AngleEmbedding, RY 회전 |
| 큐비트 수 | 8개 |
| 학습 회로 | StronglyEntanglingLayers 2층 |
| 얽힘 | CNOT 기반 |
| 측정 | 8개 큐비트의 Pauli-Z 기대값 |
| 출력층 | Linear(8 → 1) |
| 학습 파라미터 | 57개 (회로 48 + 출력층 9) |
| 손실함수 | BCEWithLogitsLoss |
| Optimizer | Adam |
| Learning rate | 0.01 |
| Batch size | 64 |
| 최대 Epoch | 100 |
| Early stopping | Patience 15 |
| 최적 Epoch | 69 |
| 최적 검증 Loss | 0.5885 |
| 시뮬레이터 | PennyLane default.qubit |
| 측정 방식 | shots=None |
| 연산 환경 | CPU, float64 |

**D. DR-VQC (8큐비트 + Data Re-uploading)**

| 단계 | 구성 |
|---|---|
| 입력 | PCA 8개 성분 |
| 큐비트 수 | 8개 |
| 데이터 재주입 횟수 | 2회 |
| 인코딩 | AngleEmbedding, RY 회전 |
| 학습 회로 | StronglyEntanglingLayers |
| 재업로딩 블록당 양자층 | 2층 |
| 전체 양자층 | 4층 |
| 측정 | 8개 큐비트의 Pauli-Z 기대값 |
| 출력층 | Linear(8 → 1) |
| 학습 파라미터 | 105개 (회로 96 + 출력층 9) |
| 손실함수 | BCEWithLogitsLoss |
| Optimizer | Adam |
| Learning rate | 0.01 |
| Batch size | 64 |
| 최대 Epoch | 100 |
| Early stopping | Patience 15 |
| 조기 종료 시점 | 34 Epoch |
| 최적 검증 Loss | 0.5621 |
| 시뮬레이터 | PennyLane default.qubit |
| 연산 환경 | CPU, float64 |

#### 3. 모델별 성능 비교

| 지표 | A. XGBoost | B. PCA-XGBoost | C. VQC | D. DR-VQC |
|---|---:|---:|---:|---:|
| Accuracy | **0.8615** | 0.8232 | 0.6990 | 0.7532 |
| Precision | **0.8429** | 0.8182 | 0.7010 | 0.7607 |
| Recall | **0.8885** | 0.8312 | 0.6943 | 0.7389 |
| F1-score | **0.8651** | 0.8246 | 0.6976 | 0.7496 |
| ROC-AUC | **0.9453** | 0.8946 | 0.7791 | 0.8313 |
| AP | **0.9447** | 0.8677 | 0.7863 | 0.8178 |
| Log Loss | **0.3044** | 0.4052 | 0.5619 | 0.5128 |

※ Precision, Recall, F1-score는 원본 타깃 코드 2를 양성 클래스로 계산.

#### 4. 핵심 결과

- 최고 성능: 16개 변수 기반 XGBoost
- PCA 적용 후 XGBoost 성능 하락
- VQC는 PCA-XGBoost보다 낮은 성능
- Data Re-uploading VQC는 기본 VQC보다 성능 향상
- DR-VQC는 기본 VQC보다 학습 파라미터가 57개에서 105개로 증가
- 양자회로의 표현력 개선 가능성을 확인했으나 고전 모델의 성능을 넘지는 못함

#### 5. 전처리 및 변수 선정 과정에서 제외된 변수

| 구분 | 변수 | 처리 방식 |
|---|---|---|
| 식별자 | 회사코드 | 모델 입력에서 제외 |
| 타깃 | 정상/폐업 | 종속변수로 사용 |
| 날짜 | 재무비율기준/결산기준일 | 모델 입력에서 제외 |
| 선택 변수 | 업력(K1) | 기존 실험에서 제외했으나 이번 실험에서는 별도 처리 후 포함 |
| 나머지 재무변수 | 16개 선정 목록 외 변수 | 모델 입력에서 제외 |
| 범주형 변수 | 산업코드, 감사의견코드 | 이번 실험에는 미포함 |

**중요한 차이점**

이번 실험에서는 업력(K1)을 다시 포함했음.

기존 309개 변수 실험에서는 업력이 예측시점 이후 정보를 포함할 가능성이 있어 제외했지만, 이번 실험에서는 16개 변수 선정 과정에서 다시 추가했음.

따라서 업력의 실제 산출 기준과 예측 기준시점을 확인해야 함. 이는 잠재적인 데이터 누수 문제임.

또한 이번 16개 변수는 자동으로 중요도 상위 16개를 추출한 것이 아니라, 코드에 명시된 변수 목록을 직접 선택한 것임.

#### 6. 결론

이번 실험에서는 16개 재무변수만 사용하더라도 XGBoost가 우수한 분류 성능을 보였다.

16개 변수를 직접 입력한 XGBoost의 ROC-AUC는 0.9453이었으며, PCA를 통해 8개 성분으로 축소한 경우 0.8946으로 하락했다.

양자 머신러닝에서는 기본 VQC보다 Data Re-uploading VQC가 더 높은 성능을 기록했다. 그러나 XGBoost 및 PCA-XGBoost의 성능을 넘지는 못했다.

따라서 현재 결과만으로는 양자 머신러닝의 성능 우위를 확인하기 어렵다. 다만 Data Re-uploading을 통한 양자회로의 표현력 개선 가능성은 추가 연구 가치가 있다.

#### 7. 추가 확인 사항

- [ ] 타깃값 원본 코드 1, 2의 실제 의미 확인
- [ ] 업력(K1)의 산출 기준 및 예측시점 이후 정보 포함 여부 확인
- [ ] 업력 포함 / 제외에 따른 성능 비교
- [ ] 랜덤 시드 변경 후 성능 재검증
- [ ] 동일한 PCA 차원에서 XGBoost와 VQC 비교
- [ ] XGBoost의 PCA 버전과 일반 버전의 하이퍼파라미터 통일
- [ ] DR-VQC 재업로딩 횟수에 따른 성능 변화 확인
- [ ] PCA 성분 수 변경에 따른 성능 변화 확인
- [ ] 결산연도에 따른 시점 교란 가능성 확인
- [ ] 시간 순서 기반 Train / Test 분할을 통한 미래 데이터 일반화 성능 검증
- [ ] 동일한 데이터 분할과 여러 랜덤 시드에서 양자회로 성능 안정성 확인

#### 8. 기존 309개 변수 실험과 비교

| 구분 | 309개 변수 실험 | 16개 변수 실험 |
|---|---:|---:|
| XGBoost Accuracy | 0.8933 | 0.8615 |
| XGBoost ROC-AUC | 0.9648 | 0.9453 |
| PCA-XGBoost Accuracy | 0.8280 | 0.8232 |
| VQC Accuracy | 0.8057 | 0.6990 |
| DR-VQC Accuracy | 0.7946 | 0.7532 |

16개 변수로 줄였을 때 XGBoost의 성능 하락보다 VQC의 성능 하락이 더 크게 나타났다.

다만 두 실험의 변수 구성과 전처리 조건이 다르므로, 이를 양자 모델의 변수 축소 취약성으로 단정할 수는 없다. 동일한 조건에서 추가 검증이 필요하다.
권오병 논문에서 양자에 유리한 데이터 추출하는 부분 참고해서 새실험 세팅